# Parse documents with the Unstructured Transform MCP server

Real-world documents — PDFs with multi-column layouts, scanned pages, nested tables, images —
are hard to feed to a language model directly. [Unstructured](https://unstructured.io) solves this
with **Transform**, a hosted service that turns 60+ file types into clean, structured content
(markdown, Element JSON, HTML, or text).

Unstructured Transform is available as a **remote [Model Context Protocol (MCP)](https://modelcontextprotocol.io)
server**, which means you can plug it straight into the OpenAI **Responses API** as a tool. The
model can then parse a document on demand and reason over the result — no bespoke parsing pipeline
to build or host.

In this recipe you will:

1. Connect the hosted Unstructured Transform MCP server to the Responses API and list its tools.
2. Have the model parse a complex, table-heavy PDF into markdown.
3. Fetch the parsed output and ask grounded questions over it.

The demo document is the classic *"Attention Is All You Need"* paper, chosen because it contains
tables and mathematical notation that trip up naive text extraction.

## 1. Prerequisites

You need two API keys, both read from environment variables (never hard-code secrets in a notebook):

- **`OPENAI_API_KEY`** — from <https://platform.openai.com/api-keys>.
- **`UNSTRUCTURED_API_KEY`** — sign up at <https://transform.unstructured.io/>, then fetch your API key from the same page. This is the Transform API key that the hosted MCP server accepts as a bearer token (see step 2).

Don't forget to set your API keys :) 

```bash
export OPENAI_API_KEY="sk-..."
export UNSTRUCTURED_API_KEY="..."
```

In [ ]:
%pip install --upgrade openai requests

In [ ]:
import json
import os
import re
import time

import requests
from openai import OpenAI
from getpass import getpass




if "UNSTRUCTURED_API_KEY" not in os.environ:
    os.environ["UNSTRUCTURED_API_KEY"] = getpass("Enter your Unstructured API key: ")
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# The hosted Unstructured Transform MCP server speaks streamable HTTP.
TRANSFORM_MCP_URL = "https://mcp.transform.unstructured.io/"
MODEL = "gpt-5"

## 2. Connect the Transform MCP server

The Responses API can call a remote MCP server directly — you just describe it as a tool.

The hosted Transform server supports two authentication paths: an interactive **OAuth** flow (used by
desktop MCP clients like Claude Desktop, e.g. via `npx mcp-remote`) and a static **bearer token**. Because
the Responses API authenticates server-to-server and can't complete a browser-based OAuth handshake, we
use the bearer-token path: pass your Unstructured API key in the `Authorization` header as
`Bearer <UNSTRUCTURED_API_KEY>`. The server accepts a valid Transform API key directly as that bearer token.

> **Note on headers:** OpenAI forwards the `headers` you provide to the MCP server but does **not**
> store them (and strips all but the domain of `server_url`) after each request. That means the tool
> block must be included on *every* `responses.create` call.

`require_approval="never"` skips the manual approval step that otherwise pauses the run before each
tool call — appropriate here because we control the server.

In [ ]:
def transform_mcp_tool(allowed_tools=None):
    """Build a Responses API MCP tool block for the Unstructured Transform server.

    Args:
        allowed_tools: Optional list of tool names to expose to the model. Restricting
            the surface area reduces prompt size and keeps the model focused.

    Returns:
        A tool dict suitable for the ``tools`` argument of ``client.responses.create``.
    """
    tool = {
        "type": "mcp",
        "server_label": "unstructured_transform",
        "server_url": TRANSFORM_MCP_URL,
        "require_approval": "never",
        "headers": {"Authorization": f"Bearer {os.environ['UNSTRUCTURED_API_KEY']}"},
    }
    if allowed_tools:
        tool["allowed_tools"] = allowed_tools
    return tool

### Give the agent a `wait_seconds` tool

Transform parses asynchronously, so the model has to poll `check_transform_status` until the job
finishes. To let it pace those checks instead of polling in a tight loop, we add a small
`wait_seconds` **function tool**. The hosted MCP tools run server-side, but a function tool pauses
the response so we can run it locally and feed the result back, which is why the parse step below
runs a short client-side loop.

In [ ]:
def wait_seconds(seconds):
    """Wait before checking the job status again, so the agent can wait out a slow job."""
    time.sleep(seconds)
    return f"Waited {seconds} seconds."


# A client-side function tool. Unlike the hosted MCP tools (which the Responses API executes
# server-side), a function tool pauses the turn so *we* can run it and feed the result back.
WAIT_TOOL = {
    "type": "function",
    "name": "wait_seconds",
    "description": "Wait the given number of seconds before checking the job status again.",
    "parameters": {
        "type": "object",
        "properties": {"seconds": {"type": "integer"}},
        "required": ["seconds"],
        "additionalProperties": False,
    },
}

The first time the model talks to an MCP server, the Responses API calls the server's `tools/list`
and records the result as an `mcp_list_tools` output item. Let's confirm connectivity by printing the
tools Transform exposes.

In [ ]:
resp = client.responses.create(
    model=MODEL,
    tools=[transform_mcp_tool()],
    input="What tools can you use from the Unstructured Transform MCP server?",
)

for item in resp.output:
    if item.type == "mcp_list_tools":
        print(f"Tools exposed by '{item.server_label}':")
        for tool in item.tools:
            print(f"  - {tool.name}")

You should see the four Transform tools:

- `request_file_upload_url` — mint a signed URL to upload a local file.
- `transform_files` — submit 1–10 inputs (public URLs or uploads) as one async job.
- `check_transform_status` — poll a job until it completes.
- `get_transform_results` — retrieve the rendered output (markdown / JSON / HTML / text).

Because our document is already reachable at a public `https://` URL, we can skip the upload step and
hand the URL straight to `transform_files`.

## 3. Parse a complex PDF

Now let the model drive the whole pipeline: submit the transform job, wait for it to finish, and
fetch the markdown. Transform runs asynchronously, so the model calls `transform_files`, then polls
`check_transform_status` (pausing with `wait_seconds` between checks) until the job is `COMPLETED`,
and finally calls `get_transform_results`.

In [ ]:
PDF_URL = "https://arxiv.org/pdf/1706.03762"  # "Attention Is All You Need"

# Cap on client-side tool round-trips (mostly wait_seconds polls). The loop exits as soon as the
# model stops emitting function calls; this bound just guarantees it can never spin forever.
MAX_TOOL_STEPS = 40

parse_resp = client.responses.create(
    model=MODEL,
    tools=[transform_mcp_tool(), WAIT_TOOL],
    input=(
        "Use the Unstructured Transform MCP server to parse the PDF at "
        f"{PDF_URL} into markdown. Submit the job with transform_files, then poll "
        "check_transform_status, calling wait_seconds(10) between each check, until the "
        "status is COMPLETED. Then call get_transform_results and report how many elements "
        "were extracted and the download URL for the parsed markdown."
    ),
)

# The model drives the hosted MCP tools server-side, but wait_seconds is a function tool, so each
# time the model calls it the turn pauses for us to run it and continue with previous_response_id.
# OpenAI discards the MCP headers after each request, so we re-send the tool block every call.
for _ in range(MAX_TOOL_STEPS):
    calls = [item for item in parse_resp.output if item.type == "function_call"]
    if not calls:
        break
    tool_outputs = []
    for call in calls:
        args = json.loads(call.arguments)
        result = wait_seconds(**args) if call.name == "wait_seconds" else "unknown tool"
        tool_outputs.append(
            {"type": "function_call_output", "call_id": call.call_id, "output": result}
        )
    parse_resp = client.responses.create(
        model=MODEL,
        tools=[transform_mcp_tool(), WAIT_TOOL],
        previous_response_id=parse_resp.id,
        input=tool_outputs,
    )

print(parse_resp.output_text)

### Fetch the rendered output

`get_transform_results` returns each file **out of band**: instead of dumping a large document into
the model's context, it returns a short-lived, signed `download_url`. This keeps the agent's context
lean. To reason over the full content ourselves, we recover that URL from the response and download
it directly.

In [ ]:
def extract_markdown_url(response):
    """Recover the Transform markdown download URL from the MCP tool-call outputs.

    ``get_transform_results`` embeds a signed ``download_url`` in its tool output. We
    scan the serialized response for it rather than depending on a specific field path.
    """
    blob = json.dumps(response.model_dump())
    matches = re.findall(
        r"https://mcp\.transform\.unstructured\.io/output/\S+?\.md[^\"\\ ]*",
        blob,
    )
    if not matches:
        raise RuntimeError("No Transform markdown download URL found in the response.")
    return matches[0]


markdown_url = extract_markdown_url(parse_resp)
download = requests.get(markdown_url, timeout=60)
download.raise_for_status()  # fail loudly if the signed URL expired or errored
parsed_markdown = download.text
print(f"Parsed markdown: {len(parsed_markdown):,} characters\n")
print(parsed_markdown[:1200])

Notice that tables come back as clean HTML `<table>` blocks embedded in the markdown, so structure
like the BLEU-score comparison table survives — exactly the kind of content that naive PDF text
extraction mangles.

## 4. Ask questions over the parsed content

With clean markdown in hand, a final Responses API call answers questions grounded in the document.
We keep the model honest by instructing it to answer only from the provided text.

In [ ]:
qa_resp = client.responses.create(
    model=MODEL,
    input=[
        {
            "role": "system",
            "content": (
                "Answer only from the provided document. "
                "Cite the table or section you used."
            ),
        },
        {
            "role": "user",
            "content": (
                "Here is a research paper parsed to markdown:\n\n"
                f"{parsed_markdown}\n\n"
                "1) What BLEU scores did the Transformer (big) model achieve on the "
                "EN-DE and EN-FR newstest2014 tasks?\n"
                "2) According to Table 1, what is the maximum path length of a "
                "self-attention layer?"
            ),
        },
    ],
)

print(qa_resp.output_text)

The model reads the answers straight out of the parsed tables — for example, Transformer (big)
scores **28.4 BLEU (EN-DE)** and **41.8 BLEU (EN-FR)** from Table 2, and a self-attention layer has a
maximum path length of **O(1)** per Table 1.

## 5. Best practices

- **Restrict the tool surface.** Pass `allowed_tools=[...]` to `transform_mcp_tool()` to expose only
  the tools a given task needs.
- **Re-send credentials every call.** OpenAI discards `headers` and all-but-domain of `server_url`
  after each request, so include the tool block on every `responses.create` call.
- **Mind the limits.** Transform accepts up to 50 MB per file, 10 files per request, and 5 concurrent
  jobs. For local files, mint an upload URL with `request_file_upload_url` first.
- **Pick your output format.** `get_transform_results` can render the same job as markdown, Element
  JSON, HTML, or plain text — Element JSON is ideal when you want typed elements and metadata for a
  downstream RAG pipeline.
- **Keep secrets in env vars.** Never hard-code API keys in a shared notebook.

## Conclusion

By exposing Unstructured Transform as a remote MCP server, the OpenAI Responses API can parse
messy, real-world documents on demand and reason over the clean result — no custom extraction
pipeline required. From here you could:

- Swap in your own PDFs, DOCX, PPTX, XLSX, emails, or images.
- Request **Element JSON** and build a retrieval index over the typed elements.
- Combine the Transform tool with other tools (web search, file search) in a single agent.

### Resources

- [Unstructured Transform overview](https://docs.unstructured.io/transform/overview)
- [Model Context Protocol](https://modelcontextprotocol.io)
